<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/FISCAL/PIM_Funcion_Municipalidad_LOR_SMN_UCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN (ejecutar una vez por sesión) ═══
!pip install requests beautifulsoup4 pandas numpy tqdm google-api-python-client --quiet
print("✅ Librerías listas")
import os, re, time, random, zipfile, unicodedata, threading
import requests
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from IPython.display import display

In [ ]:
# ═══ BLOQUE 2 · CONFIGURACIÓN, EXTRACCIÓN Y CIERRE ═══
# ── 1. CONFIGURACIÓN ─────────────────────────────────────────────────────────
# El PIM es un monto ANUAL: se consulta sin filtro de mes.
AÑO_INICIO = 2026
AÑO_FIN    = 2026

CONTROL = True                 # suma de municipalidades vs PIM de la función en el departamento
SUBIR_A_DRIVE = True
DRIVE_FOLDER_ID = "1gnXMhMyfiKz_Hy-LIrR3lhPXqXgcysxx"           # CSV de datos y LOG
DRIVE_FOLDER_CONTROL_ID = "1vIywq5uTfzu5Uw9xo9CIisg3aF7hfZ6m"   # archivos CONTROL
DESCARGA_FINAL = "ninguna"     # "zip" | "individual" | "ninguna"

HILOS = 3                      # un hilo por región (Loreto, San Martín, Ucayali)
PAUSA = (0.4, 1.0)             # segundos aleatorios entre consultas de cada hilo

# ── Cómo arma la URL Consulta Amigable ───────────────────────────────────────
# Cada filtro es una clave numérica (24=Genérica, 1=Nivel, 37=Gob.Loc./Mancom.,
# 5=Departamento, 21=Departamento (Meta), 8=Función, 7=Municipalidad). La PRIMERA
# clave con valor vacío es la columna que se lista en la tabla de detalle.
# Ruta: Departamento > Función > Municipalidad (todas las funciones listadas, también las de PIM 0).
NIVEL_GOBIERNO = "M"                          # M = Gobiernos Locales
NIVEL_ETIQUETA = "M: GOBIERNOS LOCALES"
PARAMS_NIVEL = {"37": "M"}                    # Gob.Loc./Mancom. = M: MUNICIPALIDADES
GENERICA = "6-2-6"                            # Genérica 6-26
CLAVES_DEPTO = ["5", "21"]                    # Departamento y Departamento (Meta)
CLAVE = {"Función": "8", "Municipalidad": "7"}
NIVELES = ["Función", "Municipalidad"]
COLUMNA_DE = {"Función": "FUNCION", "Municipalidad": "MUNICIPALIDAD"}
COLUMNA_VALOR = -7             # PIM: 7.ª celda desde el final (PIA | PIM | Cert. | … | Avance %)
PARAMS_EXTRA = {}   # si F12>Network muestra otros parámetros, se sobreescriben aquí

# ── Catálogos ────────────────────────────────────────────────────────────────
departamentos = [
    {"codigo": "16", "departamento": "16: LORETO"},
    {"codigo": "22", "departamento": "22: SAN MARTÍN"},
    {"codigo": "25", "departamento": "25: UCAYALI"},
]
deptos_map = {d["codigo"]: d["departamento"] for d in departamentos}
SUFIJO_REGIONES = "LOR - SMN - UCA"

hoy = date.today()
if not (2007 <= AÑO_INICIO <= AÑO_FIN <= hoy.year):
    raise ValueError(f"Rango de años inválido ({AÑO_INICIO}-{AÑO_FIN}). Máximo: {hoy.year}.")
años = list(range(AÑO_INICIO, AÑO_FIN + 1))
etiqueta_años = f"{AÑO_INICIO}" if AÑO_INICIO == AÑO_FIN else f"{AÑO_INICIO}-{AÑO_FIN}"

ruta_guardado = "/content/PIM_Funcion.Municipalidad_LOR_SMN_UCA"
os.makedirs(ruta_guardado, exist_ok=True)

def nombre_archivo_salida(año):
    return f"PIM Funcion.Municipalidad {año} - {SUFIJO_REGIONES}.csv"

COLUMNAS_SALIDA = ["AÑO", "DEPARTAMENTO", "FUNCION", "MUNICIPALIDAD", "PIM"]
print(f"Años: {', '.join(map(str, años))}  |  regiones: {SUFIJO_REGIONES}")
print(f"Ruta por departamento: Departamento > {' > '.join(NIVELES)}")

# ── 2. GOOGLE DRIVE ──────────────────────────────────────────────────────────
drive_subidos, drive_fallidos = [], []
STATUS_REINTENTABLES = {403, 429, 500, 502, 503, 504}

def conectar_drive(folder_ids):
    """Autentica y verifica permiso de Editor en cada carpeta."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.errors import HttpError
    auth.authenticate_user()
    servicio = build("drive", "v3", cache_discovery=False)
    for folder_id in dict.fromkeys(folder_ids):
        try:
            carpeta = servicio.files().get(
                fileId=folder_id, fields="id,name,mimeType,capabilities(canAddChildren)",
                supportsAllDrives=True).execute()
        except HttpError as e:
            raise PermissionError(f"Sin acceso a la carpeta de Drive {folder_id} (HTTP {e.resp.status}).") from e
        if carpeta.get("mimeType") != "application/vnd.google-apps.folder":
            raise ValueError(f"El ID {folder_id} no corresponde a una carpeta de Drive.")
        if not carpeta.get("capabilities", {}).get("canAddChildren", False):
            raise PermissionError(f"Sin permiso de Editor en '{carpeta['name']}'.")
        print(f"☁️  Drive conectado -> carpeta: '{carpeta['name']}'")
    print()
    return servicio

def subir_a_drive(servicio, ruta_local, folder_id, max_reintentos=5):
    from googleapiclient.errors import HttpError
    from googleapiclient.http import MediaFileUpload
    nombre = os.path.basename(ruta_local)
    mime = "application/zip" if nombre.lower().endswith(".zip") else "text/csv"
    nombre_q = nombre.replace("\\", "\\\\").replace("'", "\\'")
    ultimo_error = ""
    for intento in range(1, max_reintentos + 1):
        try:
            existentes = servicio.files().list(
                q=f"name = '{nombre_q}' and '{folder_id}' in parents and trashed = false",
                fields="files(id)", supportsAllDrives=True,
                includeItemsFromAllDrives=True).execute().get("files", [])
            media = MediaFileUpload(ruta_local, mimetype=mime, resumable=True)
            if existentes:
                servicio.files().update(fileId=existentes[0]["id"], media_body=media,
                                        fields="id", supportsAllDrives=True).execute()
                accion = "actualizado"
            else:
                servicio.files().create(body={"name": nombre, "parents": [folder_id]},
                                        media_body=media, fields="id",
                                        supportsAllDrives=True).execute()
                accion = "creado"
            drive_subidos.append(nombre)
            print(f"   ☁️ Drive ({accion}): {nombre}")
            return True
        except HttpError as e:
            ultimo_error = f"HTTP {e.resp.status}"
            if e.resp.status not in STATUS_REINTENTABLES:
                break
        except Exception as e:
            ultimo_error = str(e)
        if intento < max_reintentos:
            espera = 2 ** intento + random.random()
            print(f"   ↻ Reintento {intento}/{max_reintentos - 1} en {espera:.1f}s ({ultimo_error})")
            time.sleep(espera)
    drive_fallidos.append((nombre, ultimo_error))
    print(f"   ❌ No se pudo subir {nombre} ({ultimo_error}). Sigue en {ruta_local}")
    return False



# ── 3. NAVEGACIÓN EN CONSULTA AMIGABLE (solo por URL) ────────────────────────
URL_BASE = "https://apps5.mineco.gob.pe/transparencia/mensual/Navegar_6.aspx"
_local = threading.local()

def sesion():
    """Una sesión HTTP por hilo (requests.Session no es segura entre hilos)."""
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
        _local.s.headers.update({
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
            "Referer": "https://apps5.mineco.gob.pe/transparencia/mensual/default.aspx"})
    return _local.s

# Variantes de URL que se prueban una vez (calibrar) hasta que una traiga la
# tabla de detalle. La primera imita al navegador (iframe de default.aspx).
FORMATOS = [
    {"nombre": "navegador GET", "metodo": "GET",
     "pre": {"_tgt": "frame", "_uhc": "yes"}, "post": {"ap": "ActProy"}},
    {"nombre": "GET + ap", "metodo": "GET", "pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "POST + ap", "metodo": "POST", "pre": {}, "post": {"ap": "ActProy"}},
    {"nombre": "POST original", "metodo": "POST", "pre": {}, "post": {}},
]
FORMATO = None                 # lo fija calibrar()

def a_numero(txt):
    t = (txt or "").replace(",", "").replace("\xa0", "").strip()
    if t in ("", "-"):
        return 0.0
    try:
        return float(t)
    except ValueError:
        return np.nan

def normalizar(txt):
    t = unicodedata.normalize("NFKD", txt or "")
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

_FILTRO = re.compile(r"^(total|gen[eé]rica|nivel de gobierno|gob\.?\s?loc|departamento|"
                     r"provincia|distrito|funci[oó]n|mes\b|trimestre|sector|pliego|ejecutora|"
                     r"unidad ejecutora|fuente|rubro|categor|producto|divisi[oó]n|"
                     r"grupo funcional|sub-?gen[eé]rica)", re.I)
_CODIGO = re.compile(r"^([\w\-\.]+)\s*:")

class Pagina:
    """Respuesta de Consulta Amigable ya parseada.
    filtros: breadcrumb [(etiqueta, monto)]
    filas  : detalle    [(valor_para_url, etiqueta, monto)]   (monto = columna COLUMNA_VALOR)"""
    def __init__(self, html, url):
        self.url, self.html = url, html
        s = BeautifulSoup(html, "html.parser")
        self.tiene_tabla = s.find("table", class_="Data") is not None
        self.filtros, self.filas = [], []
        for tr in s.find_all("tr"):
            if tr.find("table"):
                continue
            celdas = tr.find_all("td")
            if len(celdas) < 8:
                continue
            dev = a_numero(celdas[COLUMNA_VALOR].get_text(strip=True))
            if dev != dev:                                     # NaN -> no es fila numérica
                continue
            textos = [" ".join(c.get_text(" ", strip=True).split()) for c in celdas[:3]]
            etiqueta = next((t for t in textos if t), "")
            radio = tr.find("input", attrs={"type": "radio"})
            if _FILTRO.match(etiqueta) and radio is None:
                self.filtros.append((etiqueta, dev))
                continue
            m = _CODIGO.match(etiqueta)
            valor = (radio.get("value") if radio is not None else None) or (m.group(1) if m else None)
            if valor is None:
                continue
            self.filas.append((valor.strip(), etiqueta, dev))

def armar_params(anio, mes, depto, ruta, agrupar, formato):
    """ruta = [(clave, valor), …] de filtros ya elegidos; agrupar = clave a listar."""
    p = dict(formato["pre"])
    p.update({"0": "", "24": GENERICA})
    if mes:
        p["23"] = mes                  # vista anual: se OMITE (vacío cortaría la ruta)
    p["1"] = NIVEL_GOBIERNO
    p.update(PARAMS_NIVEL)
    for k in CLAVES_DEPTO:
        p[k] = depto
    for k, v in ruta:
        p[k] = v
    p[agrupar] = ""
    p["y"] = anio
    p.update(formato["post"])
    p.update(PARAMS_EXTRA)
    return {k: v for k, v in p.items() if v is not None}

def consultar(anio, mes, depto, ruta, agrupar, formato=None, reintentos=4):
    formato = formato or FORMATO
    params = armar_params(anio, mes, depto, ruta, agrupar, formato)
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion().request(formato["metodo"], URL_BASE, params=params, timeout=40)
            if r.status_code == 200:
                p = Pagina(r.content, r.url)
                if p.tiene_tabla:
                    return p, None
                ultimo = "SIN_TABLA"
            else:
                ultimo = f"HTTP {r.status_code}"
        except requests.RequestException as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(30, 2 ** i + random.random()))
    return None, ultimo

def calibrar(anio, mes, depto, verbose=True):
    """Prueba los FORMATOS hasta que uno liste funciones del departamento."""
    global FORMATO
    for f in FORMATOS:
        p, err = consultar(anio, mes, depto, [], CLAVE["Función"], formato=f, reintentos=2)
        n = len(p.filas) if p else 0
        if verbose:
            print(f"   · {f['nombre']:<15} -> {err or f'{n} filas de detalle'}")
        if p and n and not validar_departamento(p, depto):
            FORMATO = f
            return p
        time.sleep(random.uniform(*PAUSA))
    return None

def cuadre(p, contexto):
    """Suma del detalle vs la última fila del breadcrumb (el filtro aplicado)."""
    if not p.filtros or not p.filas:
        return None
    total = p.filtros[-1][1]
    suma = sum(v for _, _, v in p.filas)
    if abs(suma - total) > max(1, len(p.filas)):
        return f"CUADRE {contexto}: suma {suma:,.0f} vs {p.filtros[-1][0][:40]} {total:,.0f}"
    return None

def validar_departamento(p, depto):
    errores = []
    etiquetas = [normalizar(e) for e, _ in p.filtros]
    if not any(e.startswith("nivel de gobierno") and f" {NIVEL_GOBIERNO.lower()}:" in e for e in etiquetas):
        errores.append(f"la página no está filtrada por {NIVEL_ETIQUETA}")
    if not any(e.startswith("departamento") and f"{depto}:" in e for e in etiquetas):
        errores.append(f"la página no corresponde al departamento {depto}")
    return errores

USAR_CODIGO_ETIQUETA = set()   # claves donde el código de la etiqueta funcionó mejor que el radio

def codigos_de(etiqueta):
    """Códigos posibles de una etiqueta: '001-861: REGION LORETO…' -> ['001-861', '861']."""
    m = _CODIGO.match(etiqueta)
    if not m:
        return []
    cod = m.group(1)
    return [cod] + ([cod.split("-")[-1]] if "-" in cod else [])

def abrir_nivel(anio, mes, depto, ruta, nivel, candidatos=(None,), esperado=None):
    """Consulta la lista de `nivel` bajo `ruta`. Si la última fila elegida se
    puede expresar con varios códigos (valor del radio o código de la etiqueta),
    prueba cada uno hasta que el breadcrumb confirme que el filtro se aplicó."""
    ultimo_err = ""
    for cand in candidatos:
        r = list(ruta) if cand is None else list(ruta[:-1]) + [(ruta[-1][0], cand)]
        p, err = consultar(anio, mes, depto, r, CLAVE[nivel])
        time.sleep(random.uniform(*PAUSA))
        if err:
            ultimo_err = err
            continue
        if esperado is None:
            return p, r, None
        fin = p.filtros[-1][0] if p.filtros else ""
        if normalizar(esperado) in normalizar(fin):
            if cand is not None and cand != candidatos[0]:
                USAR_CODIGO_ETIQUETA.add(ruta[-1][0])
            return p, r, None
        ultimo_err = (f"la URL no aplicó el filtro '{esperado[:40]}' "
                      f"(breadcrumb termina en '{fin[:50]}'; código probado: {cand})")
    return None, None, ultimo_err

def recorrer(anio, mes, depto, niveles, ruta=(), nombres=(), candidatos=(None,)):
    """Baja nivel por nivel. Devuelve (hojas, errores, avisos);
    hojas = [(nombres_de_la_ruta, etiqueta, devengado)] del último nivel."""
    nivel = niveles[0]
    contexto = " > ".join(nombres) or "departamento"
    p, ruta, err = abrir_nivel(anio, mes, depto, ruta, nivel, candidatos,
                               nombres[-1] if nombres else None)
    if err:
        return [], [f"{nivel} en {contexto}: {err}"], []
    if not p.filas:
        if p.filtros and p.filtros[-1][1] == 0:
            return [], [], []
        return [], [f"{nivel} en {contexto}: sin filas de detalle"], []
    avisos = [a for a in [cuadre(p, f"{nivel} en {contexto}")] if a]
    if len(niveles) == 1:
        return [(nombres, et, dev) for _, et, dev in p.filas], [], avisos
    hojas, errores = [], []
    for valor, etiqueta, dev in p.filas:  # también las filas en 0: el MEF las lista y se conservan
        cands = [valor] + codigos_de(etiqueta)
        if CLAVE[nivel] in USAR_CODIGO_ETIQUETA:
            cands.reverse()
        cands = tuple(dict.fromkeys(cands))
        h, e, a = recorrer(anio, mes, depto, niveles[1:], tuple(ruta) + ((CLAVE[nivel], valor),),
                           nombres + (etiqueta,), cands)
        hojas += h; errores += e; avisos += a
    return hojas, errores, avisos

def extraer(anio, mes, depto, niveles=NIVELES):
    """-> (registros, error, aviso) para un departamento y un año."""
    if FORMATO is None and calibrar(anio, mes, depto, verbose=False) is None:
        return [], "CALIBRACIÓN: ningún formato de URL trae la tabla de detalle", None
    hojas, errores, avisos = recorrer(anio, mes, depto, niveles)
    regs = []
    for nombres, etiqueta, dev in hojas:
        r = {"AÑO": anio, "NIVEL_GOBIERNO": NIVEL_ETIQUETA,
             "DEPARTAMENTO": deptos_map[depto], "COD_DEPTO": depto, "PIM": dev}
        for n, valor in zip(niveles[:-1], nombres):
            r[COLUMNA_DE[n]] = valor
        r[COLUMNA_DE[niveles[-1]]] = etiqueta
        m = _CODIGO.match(r["FUNCION"])
        r["COD_FUNCION"] = m.group(1) if m else r["FUNCION"][:2]
        regs.append(r)
    return (regs, ("; ".join(errores)) if errores else None,
            ("; ".join(avisos) if avisos else None))

# ── 4. CONEXIONES ─────────────────────────────────────────────────────────────
carpetas_drive = [f for f in (DRIVE_FOLDER_ID, DRIVE_FOLDER_CONTROL_ID) if f]
drive = conectar_drive(carpetas_drive) if SUBIR_A_DRIVE and carpetas_drive else None
SUBIR_DATOS = bool(drive and DRIVE_FOLDER_ID)
if not SUBIR_DATOS and DESCARGA_FINAL == "ninguna":
    DESCARGA_FINAL = "zip"     # sin carpeta de datos en Drive: se descargan para no perderlos

# Elige el formato de URL que devuelve la tabla de detalle (1 a 4 consultas)
FORMATO = None
if calibrar(años[0], "", departamentos[0]["codigo"], verbose=False) is None:
    raise RuntimeError("Consulta Amigable no devolvió la tabla de detalle con ningún formato de URL.")
print(f"🔗 Consulta Amigable OK (formato: {FORMATO['nombre']})\n")

def extraer_departamento(anio, depto):
    """Municipalidades x función + referencia por función (para el control)."""
    regs, err, aviso = extraer(anio, "", depto)
    ref, err_ref = [], None
    if CONTROL:
        ref, err_ref, _ = extraer(anio, "", depto, niveles=["Función"])
    return regs, err, aviso, ref, err_ref

# ── 5. EXTRACCIÓN POR AÑO (departamentos en paralelo) ────────────────────────
errores_totales, avisos_totales, controles = [], [], []
archivos_generados, resumen = [], []

for anio in años:
    regs_anio, errs_anio = [], []
    with ThreadPoolExecutor(max_workers=HILOS) as ex:
        futuros = {ex.submit(extraer_departamento, anio, d["codigo"]): d for d in departamentos}
        for fut in tqdm(as_completed(futuros), total=len(futuros), desc=f"PIM {anio}"):
            d = futuros[fut]
            regs, err, aviso, ref, err_ref = fut.result()
            regs_anio.extend(regs)
            if err:
                errs_anio.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"], "ERROR": err})
            if aviso:
                avisos_totales.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"], "AVISO": aviso})
            if err_ref:
                errs_anio.append({"AÑO": anio, "DEPARTAMENTO": d["departamento"],
                                  "ERROR": f"Referencia del control: {err_ref}"})
            # Control: PIM de cada función en el departamento vs suma de sus municipalidades
            if CONTROL and ref:
                sub = pd.DataFrame(regs) if regs else pd.DataFrame(columns=["COD_FUNCION", "PIM"])
                for r in ref:
                    filas_f = sub[sub["COD_FUNCION"] == r["COD_FUNCION"]]
                    n_mun = len(filas_f)
                    suma = float(filas_f["PIM"].sum())
                    tol = max(3, int(np.ceil(0.5 * n_mun)))   # redondeo: hasta S/ 0.5 por municipalidad
                    controles.append({
                        "AÑO": anio, "DEPARTAMENTO": d["departamento"],
                        "COD_FUNCION": r["COD_FUNCION"], "FUNCION": r["FUNCION"],
                        "PIM_REF_FUNCION": r["PIM"], "PIM_SUMA_MUNICIPALIDADES": suma,
                        "DIF": suma - r["PIM"], "N_MUNICIPALIDADES": n_mun,
                        "ESTADO": "OK" if abs(suma - r["PIM"]) <= tol else "REVISAR",
                        "TOLERANCIA_S/": tol, "FECHA_EXTRACCION": date.today().isoformat()})

    errores_totales.extend(errs_anio)
    df_anio = pd.DataFrame(regs_anio)
    if df_anio.empty:
        estado = "SIN DATOS"
    else:
        df_anio = df_anio.sort_values(["COD_DEPTO", "COD_FUNCION", "MUNICIPALIDAD"])
        archivo = os.path.join(ruta_guardado, nombre_archivo_salida(anio))
        df_anio[COLUMNAS_SALIDA].to_csv(archivo, index=False, encoding="utf-8-sig")
        archivos_generados.append(archivo)
        if SUBIR_DATOS:
            subir_a_drive(drive, archivo, DRIVE_FOLDER_ID)
        estado = "OK" if not errs_anio else "INCOMPLETO (con errores)"
    resumen.append({"AÑO": anio, "FILAS": len(df_anio), "ERRORES": len(errs_anio), "ESTADO": estado})

print("═" * 70)
print("RESUMEN PIM MUNICIPALIDADES")
display(pd.DataFrame(resumen))
if errores_totales:
    print("⚠️ Consultas con error:")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(errores_totales))
if avisos_totales:
    print("⚠️ Avisos de cuadre interno (suma del detalle ≠ fila del filtro):")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(avisos_totales))
if not errores_totales and not avisos_totales:
    print("✅ Sin errores y con cuadre interno en todas las consultas.")

# ── 6. CONTROL + CIERRE (guardado, Drive, descarga) ──────────────────────────
if CONTROL and controles:
    ctl = pd.DataFrame(controles).sort_values(["AÑO", "DEPARTAMENTO", "COD_FUNCION"])
    print("\n🔍 Control: suma de municipalidades vs PIM de la función en el departamento")
    display(ctl["ESTADO"].value_counts().to_frame("funciones"))
    if (ctl["ESTADO"] == "REVISAR").any():
        display(ctl[ctl["ESTADO"] == "REVISAR"])
    archivo_ctl = os.path.join(ruta_guardado,
                               f"CONTROL PIM Funcion.Municipalidad {etiqueta_años} - {SUFIJO_REGIONES}.csv")
    ctl.to_csv(archivo_ctl, index=False, encoding="utf-8-sig")
    archivos_generados.append(archivo_ctl)
    if drive:
        subir_a_drive(drive, archivo_ctl, DRIVE_FOLDER_CONTROL_ID)
elif CONTROL:
    print("\n⚠️ No se pudo armar el control (sin referencia del MEF).")

if errores_totales or avisos_totales:
    log = pd.concat([pd.DataFrame(errores_totales), pd.DataFrame(avisos_totales)], ignore_index=True)
    archivo_log = os.path.join(ruta_guardado,
                               f"LOG PIM Funcion.Municipalidad {etiqueta_años} - {SUFIJO_REGIONES}.csv")
    log.to_csv(archivo_log, index=False, encoding="utf-8-sig")
    archivos_generados.append(archivo_log)
    if SUBIR_DATOS:
        subir_a_drive(drive, archivo_log, DRIVE_FOLDER_ID)

if drive:
    print(f"\n☁️ DRIVE: {len(drive_subidos)} subido(s), {len(drive_fallidos)} fallido(s)")
    if SUBIR_DATOS:
        print(f"   Datos:   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
    print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")

if not archivos_generados:
    print("\n⚠️ No hay archivos para descargar.")
elif DESCARGA_FINAL == "zip":
    from google.colab import files
    ruta_zip = os.path.join(ruta_guardado, f"PIM Funcion.Municipalidad {etiqueta_años} - {SUFIJO_REGIONES}.zip")
    with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for a in archivos_generados:
            z.write(a, arcname=os.path.basename(a))
    files.download(ruta_zip)
elif DESCARGA_FINAL == "individual":
    from google.colab import files
    for a in archivos_generados:
        files.download(a)
else:
    print(f"\n💾 Archivos guardados en {ruta_guardado}")